# Evaluate Performance of Qwen3-8B-AWQ

In [2]:
import tqdm
import torch
from torch import nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from functools import partial
import gc
import os
os.environ['https_proxy'] = 'http://192.168.1.12:7891'

debug = True

if debug:
    # improve torch tensor printing
    import torch
    def custom_repr(self):
        return f'{{Tensor:{tuple(self.shape)}}} {original_repr(self)}'
    original_repr = torch.Tensor.__repr__
    torch.Tensor.__repr__ = custom_repr

Here we use wikitext-2 dataset for perplexity evaluation. The dataset is automatically downloaded by the code.

In [3]:
print("Loding wikitext datasets ...")
testenc = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test', cache_dir="~/.cache/huggingface/datasets")
print("Done")

Loding wikitext datasets ...
Done


In [4]:
def evaluate(model, testenc, tokenizer):
    # we control the text length to avoid error posed by tiktoken
    testenc = tokenizer("\n\n".join(testenc['text']), return_tensors='pt')
    testenc = testenc.input_ids.to(model.device)
    nsamples = 40
    model = model.eval()

    nlls = []
    for i in tqdm.tqdm(range(nsamples), desc="evaluating Qwen on wikitext"):
        batch = testenc[:, (i * 1024):((i + 1) * 1024)].to(model.device)
        with torch.no_grad():
            lm_logits = model(batch).logits
        shift_logits = lm_logits[:, :-1, :].contiguous().float()
        shift_labels = testenc[:, (i * 1024):((i + 1) * 1024)][:, 1:]
        loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        neg_log_likelihood = loss.float() * 1024
        nlls.append(neg_log_likelihood)

    return torch.exp(torch.stack(nlls).sum() / (nsamples * 1024))

def get_model_size(model: nn.Module, data_width=16, group_size=-1):
    # store the quantization parameters: 1 fp16 scaling factors and 1 int4 zero point
    # this might not be precise
    if group_size != -1:
        data_width += (16 + 4) / group_size
    num_elements = 0
    for param in model.parameters():
        num_elements += param.numel()
    return num_elements * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

# Evaluate the performance of FP16 Qwen
## PPL

In [5]:

model_name = "Qwen/Qwen3-8B-AWQ"

In [6]:

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda"
)

/root/workspace/qwen_cpu_deployment/.venv/lib/python3.10/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.47s/it]


In [7]:
fp32_perplexity = evaluate(model, testenc, tokenizer)
print(f"\nmodel perplexity: {fp32_perplexity:.2f}")


Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors
evaluating Qwen on wikitext: 100%|██████████| 40/40 [00:05<00:00,  6.81it/s]



model perplexity: 11.52


In [ ]:
model_size = get_model_size(model, data_width=32, group_size=-1)
print(f"model size: {model_size/MiB:.2f} MiB")

model size: 4749.18 MiB


: 

In [10]:
del model
gc.collect()
torch.cuda.empty_cache()

## GSM8k

In [12]:
!lm-eval --tasks gsm8k --model vllm --model_args pretrained=Qwen/Qwen3-8B,max_model_len=8192,dtype=float16 --batch_size auto --trust_remote_code

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


INFO 05-30 09:29:10 [__init__.py:243] Automatically detected platform cuda.
2025-05-30:09:29:13 INFO     [__main__:428] Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`
2025-05-30:09:29:13 INFO     [__main__:440] Selected Tasks: ['gsm8k']
2025-05-30:09:29:13 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-05-30:09:29:13 INFO     [evaluator:223] Initializing vllm model, with arguments: {'pretrained': 'Qwen/Qwen3-8B', 'max_model_len': 8192, 'dtype': 'float16', 'trust_remote_code': True}
INFO 05-30 09:29:13 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 05-30 09:29:13 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 05-30 09:29:13 [__init__.py:36] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins

# Evaluate Performance Of Int4 Weight-quantized Model
Apply pseudo quantization to check the performance of quantized model directly

In [5]:
def pseudo_quantize_tensor_q4_zero_point(w, n_bit=4, q_group_size = -1):
    assert q_group_size == 32, "In this code, the quantize shape should be set to 32 to be compatible with tinychatengine code"

    org_w_shape = w.shape
    if q_group_size > 0:
        assert org_w_shape[-1] % q_group_size == 0
        w = w.reshape(-1, q_group_size)
        
    # get the max and min value of each group
    max_val = torch.amax(w, dim=-1, keepdim=True)
    min_val = torch.amin(w, dim=-1, keepdim=True)
    max_int = 2 ** n_bit - 1
    min_int = 0

    # get the scaling factor, zero point
    scaling_factor = (max_val - min_val).clamp(min=1e-5) / (max_int - min_int) # (len, 1)
    zero_point = (-torch.round(min_val / scaling_factor)).clamp_(0, max_int)
    # make sure no nan occurs
    assert (not scaling_factor.isnan().any())
    assert (not zero_point.isnan().any())
    
    # start the process of pseudo quantization
    # 1. quantize
    w_q =  torch.round( w / scaling_factor) + zero_point
    w_q = w_q.clamp(min_int, max_int)
    assert w_q.dim() == 2 and w_q.size(1) == q_group_size and w_q.size(0) == scaling_factor.size(0), \
            f"{w_q.size()} != {scaling_factor.size()}"
    # 2. dequantize
    w_f = (w_q - zero_point) * scaling_factor
    assert w_f.size() == w.size()
    assert (not w_f.isnan().any())
    max_error = (w_f - w).abs().max()
    w_f = w_f.reshape(org_w_shape)
    # print(f"Pseudo quantization error: {max_error}")
    return w_f 

def pseudo_quantize_tensor_q40(w, n_bit=4, q_group_size = -1):
    assert q_group_size == 32, "In this code, the quantize shape should be set to 32 to be compatible with tinychatengine code"

    org_w_shape = w.shape
    if q_group_size > 0:
        assert org_w_shape[-1] % q_group_size == 0
        w = w.reshape(-1, q_group_size)
        
    # get the max and min value of each group
    max_abs_value = torch.amax(w.abs(), dim=-1, keepdim=True)
    max_int = (2 ** (n_bit - 1) - 1)
    min_int = - (2 ** (n_bit - 1) )

    # get the scaling factor, zero point
    scaling_factor = (max_abs_value).clamp(min=1e-5) / (min_int) # (len, 1)
    zero_point = torch.tensor(8).round()
    # make sure no nan occurs
    assert (not scaling_factor.isnan().any())
    assert (not zero_point.isnan().any())
    
    # start the process of pseudo quantization
    # 1. quantize
    w_q =  torch.round( w / scaling_factor)
    w_q = w_q.clamp(min_int, max_int)
    assert w_q.dim() == 2 and w_q.size(1) == q_group_size and w_q.size(0) == scaling_factor.size(0), \
            f"{w_q.size()} != {scaling_factor.size()}"
    # 2. dequantize
    w_f = (w_q) * scaling_factor
    assert w_f.size() == w.size()
    assert (not w_f.isnan().any())
    max_error = (w_f - w).abs().max()
    w_f = w_f.reshape(org_w_shape)
    # print(f"Pseudo max quantization error: {max_error}")
    return w_f

def pseudo_quantize_tensor_q41(w, n_bit=4, q_group_size = -1):
    assert q_group_size == 32, "In this code, the quantize shape should be set to 32 to be compatible with tinychatengine code"

    org_w_shape = w.shape
    if q_group_size > 0:
        assert org_w_shape[-1] % q_group_size == 0
        w = w.reshape(-1, q_group_size)
        
    # get the max and min value of each group
    max_val = torch.amax(w, dim=-1, keepdim=True)
    min_val = torch.amin(w, dim=-1, keepdim=True)
    max_int = 2 ** n_bit - 1
    min_int = 0

    # get the scaling factor, zero point
    scaling_factor = (max_val - min_val).clamp(min=1e-5) / (max_int - min_int) # (len, 1)
    # make sure no nan occurs
    assert (not scaling_factor.isnan().any())
    
    # start the process of pseudo quantization
    # 1. quantize
    w_q = torch.round( 
            (w - min_val) / scaling_factor
        )
    w_q = w_q.clamp(min_int, max_int)
    assert w_q.dim() == 2 and w_q.size(1) == q_group_size and w_q.size(0) == scaling_factor.size(0), \
            f"{w_q.size()} != {scaling_factor.size()}"
    # 2. dequantize
    w_f = (w_q) * scaling_factor + min_val
    assert w_f.size() == w.size()
    assert (not w_f.isnan().any())
    max_error = (w_f - w).abs().max()
    w_f = w_f.reshape(org_w_shape)
    # print(f"Pseudo quantization error: {max_error}")
    return w_f 

quantization_method_dict = {
    "q4z": pseudo_quantize_tensor_q4_zero_point,
    "q40": pseudo_quantize_tensor_q40,
    "q41": pseudo_quantize_tensor_q41
}

@torch.no_grad()
def pseudo_quantize_model_weight(model, w_bit, q_group_size, method):
    q_method = quantization_method_dict[method]
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear):
            # print(f"Pseudo quantizing {n}")
            m.weight.data = q_method(m.weight.data, w_bit, q_group_size)
            # print("")

def quantize_and_evaluate(q_method):

    # load the tokenizer and the model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    # Apply fake quantization
    pseudo_quantize_model_weight(model, 4, 32, q_method)
    # Evaluate the model
    model_perplexity = evaluate(model, testenc, tokenizer)
    model_size = get_model_size(model, data_width=4, group_size=32)
    print(f"\nmodel perplexity: {model_perplexity:.2f}")
    print(f"model size: {model_size/MiB:.2f} MiB")
    return model, tokenizer


## W41

In [11]:
gc.collect()
torch.cuda.empty_cache()
model, tokenizer = quantize_and_evaluate("q41")
quantized_model_path = "tmp_quantized_model"
model.save_pretrained(quantized_model_path)
tokenizer.save_pretrained(quantized_model_path)

del model
gc.collect()
torch.cuda.empty_cache()

Loading checkpoint shards: 100%|██████████| 5/5 [00:08<00:00,  1.65s/it]
Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors
evaluating Qwen on wikitext: 100%|██████████| 40/40 [00:04<00:00,  8.75it/s]



model perplexity: 12.08
model size: 4515.90 MiB


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


INFO 05-30 08:38:31 [__init__.py:243] Automatically detected platform cuda.
2025-05-30:08:38:38 INFO     [__main__:428] Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`
2025-05-30:08:38:38 INFO     [__main__:440] Selected Tasks: ['gsm8k']
2025-05-30:08:38:38 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-05-30:08:38:38 INFO     [evaluator:223] Initializing vllm model, with arguments: {'pretrained': 'tmp_quantized_model', 'max_model_len': 8192, 'trust_remote_code': True}
INFO 05-30 08:38:38 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 05-30 08:38:38 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 05-30 08:38:38 [__init__.py:36] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO

In [ ]:

!lm-eval --tasks gsm8k --model vllm --model_args pretrained=tmp_quantized_model,max_model_len=8192 --batch_size auto --trust_remote_code

## W4z

In [6]:
gc.collect()
torch.cuda.empty_cache()
model, tokenizer = quantize_and_evaluate("q4z")
quantized_model_path = "tmp_quantized_model"
model.save_pretrained(quantized_model_path)
tokenizer.save_pretrained(quantized_model_path)

del model
gc.collect()
torch.cuda.empty_cache()

Loading checkpoint shards: 100%|██████████| 5/5 [00:01<00:00,  2.68it/s]
Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors
evaluating Qwen on wikitext: 100%|██████████| 40/40 [00:04<00:00,  8.26it/s]



model perplexity: 11.49
model size: 4515.90 MiB


In [13]:

!lm-eval --tasks gsm8k --model vllm --model_args pretrained=tmp_quantized_model,max_model_len=8192 --batch_size auto --trust_remote_code

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


INFO 05-30 09:30:37 [__init__.py:243] Automatically detected platform cuda.
2025-05-30:09:30:40 INFO     [__main__:428] Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`
2025-05-30:09:30:40 INFO     [__main__:440] Selected Tasks: ['gsm8k']
2025-05-30:09:30:40 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-05-30:09:30:40 INFO     [evaluator:223] Initializing vllm model, with arguments: {'pretrained': 'tmp_quantized_model', 'max_model_len': 8192, 'trust_remote_code': True}
INFO 05-30 09:30:40 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 05-30 09:30:40 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 05-30 09:30:40 [__init__.py:36] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO

## W40

In [6]:
gc.collect()
torch.cuda.empty_cache()
model, tokenizer = quantize_and_evaluate("q40")
quantized_model_path = "tmp_quantized_model"
model.save_pretrained(quantized_model_path)
tokenizer.save_pretrained(quantized_model_path)

del model
gc.collect()
torch.cuda.empty_cache()

Loading checkpoint shards: 100%|██████████| 5/5 [00:01<00:00,  3.06it/s]
Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors
evaluating Qwen on wikitext: 100%|██████████| 40/40 [00:04<00:00,  8.28it/s]



model perplexity: 11.73
model size: 4515.90 MiB


In [2]:

!lm-eval --tasks gsm8k --model vllm --model_args pretrained=tmp_quantized_model,max_model_len=8192 --batch_size auto --trust_remote_code

INFO 05-30 09:42:56 [__init__.py:243] Automatically detected platform cuda.
2025-05-30:09:42:59 INFO     [__main__:428] Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`
2025-05-30:09:42:59 INFO     [__main__:440] Selected Tasks: ['gsm8k']
2025-05-30:09:42:59 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-05-30:09:42:59 INFO     [evaluator:223] Initializing vllm model, with arguments: {'pretrained': 'tmp_quantized_model', 'max_model_len': 8192, 'trust_remote_code': True}
INFO 05-30 09:42:59 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 05-30 09:42:59 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 05-30 09:42:59 [__init__.py:36] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO

In [3]:
model = AutoModelForCausalLM.from_pretrained(
    "tmp_quantized_model",
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.92it/s]
